# EA3 – Comparación climática: Medellín vs. Madrid (2019)

Kevin Daniel Sepúlveda Giraldo · Analítica de Datos · IU Digital de Antioquia

**Pregunta guía:** ¿qué similitudes y diferencias hay en la temperatura y la lluvia de Medellín y Madrid durante el mismo periodo?

## A. Planteamiento

### Fuentes

| | Medellín | Madrid |
|---|---|---|
| Fuente | NOAA NCEI – GHCN-Daily | NOAA NCEI – GHCN-Daily |
| Estación | `COM00080110` Aeropuerto Olaya Herrera | `SP000003195` Madrid Retiro |
| País | Colombia | España |
| Periodo | 2019-01-01 a 2019-12-31 | 2019-01-01 a 2019-12-31 |
| Frecuencia | diaria | diaria |
| Archivo | `datos/medellin.csv` | `datos/madrid.csv` |

Enlace del servicio: https://www.ncei.noaa.gov/access/services/data/v1 (dataset `daily-summaries`, `units=metric`).
Documentación de GHCN-Daily: https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily

Columnas que uso:

| Columna | Qué es | Unidad |
|---|---|---|
| `DATE` | fecha de la observación | AAAA-MM-DD |
| `TAVG` | temperatura media diaria | °C |
| `TMIN` | temperatura mínima diaria | °C |
| `PRCP` | precipitación diaria | mm |

Elegí la misma fuente para las dos ciudades para que el formato, las columnas y las unidades sean iguales. GHCN guarda la temperatura en décimas de °C y la lluvia en décimas de mm; al pedir los datos con `units=metric` el servicio ya los entrega en °C y mm, así que no tuve que convertir. `TMAX` no la uso porque en Medellín casi no tiene datos en 2019. Elegí 2019 porque es el año reciente con más días completos en la estación de Medellín.

Son datos de **una estación** por ciudad, no un promedio de toda la ciudad.

### Predicción (antes de calcular)

Espero que Medellín sea más cálida que Madrid todos los meses y con muy poca variación, que llueva mucho más en Medellín y que Madrid tenga un invierno frío y un verano caliente y seco.

In [ ]:
import csv
import urllib.request
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ANIO = 2019
FECHA_INICIAL = f"{ANIO}-01-01"
FECHA_FINAL = f"{ANIO}-12-31"
CARPETA_DATOS = Path("datos")

ESTACIONES = {
    "Medellín": {"id": "COM00080110", "archivo": CARPETA_DATOS / "medellin.csv"},
    "Madrid": {"id": "SP000003195", "archivo": CARPETA_DATOS / "madrid.csv"},
}

UMBRAL_LLUVIA_MM = 1.0   # día lluvioso: 1 mm o más
UMBRAL_TEMP_C = 25.0     # día cálido: temperatura media mayor a 25 °C
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

Los CSV ya están en `datos/`. Si falta alguno, esta celda lo descarga de NOAA con la misma consulta que usé.

In [ ]:
URL_NOAA = (
    "https://www.ncei.noaa.gov/access/services/data/v1"
    "?dataset=daily-summaries&dataTypes=PRCP,TAVG,TMAX,TMIN&units=metric&format=csv"
    "&startDate={inicio}&endDate={fin}&stations={estacion}"
)


def descargar_si_falta(estacion, ruta):
    if ruta.exists():
        return "ya estaba"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    url = URL_NOAA.format(inicio=FECHA_INICIAL, fin=FECHA_FINAL, estacion=estacion)
    urllib.request.urlretrieve(url, ruta)
    return "descargado"


for ciudad, info in ESTACIONES.items():
    print(ciudad, "->", info["archivo"], descargar_si_falta(info["id"], info["archivo"]))

## B. Carga y revisión

In [ ]:
def leer_csv(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        lector = csv.reader(archivo)
        encabezado = next(lector)
        filas = [fila for fila in lector if fila]  # salto líneas vacías
    return encabezado, filas


crudos = {ciudad: leer_csv(info["archivo"]) for ciudad, info in ESTACIONES.items()}

for ciudad, (encabezado, filas) in crudos.items():
    print(f"--- {ciudad} ---")
    print("Columnas:", encabezado)
    print("Dimensiones:", len(filas), "filas x", len(encabezado), "columnas")
    for fila in filas[:5]:
        print(fila)
    print()

Un campo vacío en el CSV significa que la estación no reportó ese dato. Lo convierto a `np.nan` y **no** a 0: un 0 en `PRCP` sí es un día sin lluvia reportado, pero un vacío es un dato que no existe.

In [ ]:
def a_float(valor):
    valor = valor.strip()
    if valor == "":
        return np.nan
    return float(valor)


def preparar(encabezado, filas):
    i = {nombre: encabezado.index(nombre) for nombre in ["DATE", "PRCP", "TAVG", "TMIN"]}
    fechas = np.array([datetime.strptime(f[i["DATE"]], "%Y-%m-%d") for f in filas])
    return {
        "fecha": fechas,
        "mes": np.array([d.month for d in fechas]),
        "prcp": np.array([a_float(f[i["PRCP"]]) for f in filas]),
        "tavg": np.array([a_float(f[i["TAVG"]]) for f in filas]),
        "tmin": np.array([a_float(f[i["TMIN"]]) for f in filas]),
    }


datos = {ciudad: preparar(*crudos[ciudad]) for ciudad in ESTACIONES}

for ciudad, d in datos.items():
    print(f"{ciudad}: {d['fecha'][0]:%Y-%m-%d} a {d['fecha'][-1]:%Y-%m-%d} | "
          f"tipos: fecha={type(d['fecha'][0]).__name__}, prcp={d['prcp'].dtype}, tavg={d['tavg'].dtype}")

In [ ]:
print(f"{'Ciudad':10}{'días':>6}{'duplicadas':>12}{'falta PRCP':>12}{'falta TAVG':>12}{'falta TMIN':>12}")
for ciudad, d in datos.items():
    n = len(d["fecha"])
    duplicadas = n - len(np.unique(d["fecha"]))
    print(f"{ciudad:10}{n:>6}{duplicadas:>12}"
          f"{np.isnan(d['prcp']).sum():>12}{np.isnan(d['tavg']).sum():>12}{np.isnan(d['tmin']).sum():>12}")

In [ ]:
med, mad = datos["Medellín"], datos["Madrid"]

misma_fecha = np.array_equal(med["fecha"], mad["fecha"])
print("¿Las dos tablas tienen exactamente las mismas fechas y en el mismo orden?", misma_fecha)

prcp_validas = ~np.isnan(med["prcp"]) & ~np.isnan(mad["prcp"])
tavg_validas = ~np.isnan(med["tavg"]) & ~np.isnan(mad["tavg"])
print("Días con PRCP válida en las dos ciudades:", prcp_validas.sum(), "de", len(med["fecha"]))
print("Días con TAVG válida en las dos ciudades:", tavg_validas.sum(), "de", len(med["fecha"]))

Las dos tablas cubren los 365 días de 2019, sin fechas duplicadas. El problema está en la lluvia de Medellín: faltan 71 días (casi 1 de cada 5). Para comparar día a día uso solo las fechas válidas en ambas ciudades (293 para lluvia y 356 para temperatura media).

In [ ]:
faltantes_mes = np.array([[np.isnan(d["prcp"][d["mes"] == m]).sum() for m in range(1, 13)] for d in (med, mad)])
print("Días sin PRCP por mes")
print("         ", " ".join(f"{m:>4}" for m in MESES))
for ciudad, fila in zip(ESTACIONES, faltantes_mes):
    print(f"{ciudad:9}", " ".join(f"{v:>4}" for v in fila))

En Medellín los faltantes se concentran en mayo (12 días seguidos, del 19 al 30) y julio (13). Los totales de lluvia de esos meses van a quedar por debajo de lo real.

## C. Análisis con NumPy

### Índices y cortes: primera semana del año

In [ ]:
print("Medellín TAVG 1–7 ene:", med["tavg"][:7])
print("Madrid   TAVG 1–7 ene:", mad["tavg"][:7])
print("Medellín PRCP 1–7 ene:", med["prcp"][:7])
print("Madrid   PRCP 1–7 ene:", mad["prcp"][:7])

Mismo corte en verano (1–7 de julio = posiciones 181 a 187):

In [ ]:
inicio_julio = np.where((med["mes"] == 7))[0][0]
corte = slice(inicio_julio, inicio_julio + 7)
print("Fechas:", [f"{d:%d-%b}" for d in med["fecha"][corte]])
print("Medellín TAVG:", med["tavg"][corte])
print("Madrid   TAVG:", mad["tavg"][corte])

En enero Medellín está casi 20 °C por encima de Madrid; en julio Madrid ya la supera. El `nan` del 2 de julio en Madrid es un faltante real.

### Máscaras: días lluviosos y días cálidos

In [ ]:
resumen = {}
for ciudad, d in datos.items():
    lluvioso = d["prcp"] >= UMBRAL_LLUVIA_MM   # nan >= 1 da False, así que un faltante no cuenta como lluvioso
    calido = d["tavg"] > UMBRAL_TEMP_C
    resumen[ciudad] = {
        "dias_lluviosos": int(lluvioso.sum()),
        "dias_calidos": int(calido.sum()),
        "calido_y_lluvioso": int((lluvioso & calido).sum()),
    }
    print(f"{ciudad}: {lluvioso.sum()} días con >= {UMBRAL_LLUVIA_MM} mm | "
          f"{calido.sum()} días con TAVG > {UMBRAL_TEMP_C} °C | "
          f"{(lluvioso & calido).sum()} cálidos y lluviosos a la vez")

In [ ]:
calido_med = med["tavg"] > UMBRAL_TEMP_C
calido_mad = mad["tavg"] > UMBRAL_TEMP_C
print("Meses con días cálidos en Madrid:", np.unique(mad["mes"][calido_mad]))
print("Meses con días cálidos en Medellín:", np.unique(med["mes"][calido_med]))

### Medidas descriptivas por ciudad

Uso las funciones `nan...` de NumPy para que los faltantes no dañen los cálculos.

In [ ]:
print(f"{'':28}{'Medellín':>10}{'Madrid':>10}")
filas = [
    ("TAVG promedio (°C)", np.nanmean),
    ("TAVG mínima (°C)", np.nanmin),
    ("TAVG máxima (°C)", np.nanmax),
    ("TAVG desviación estándar", np.nanstd),
]
for nombre, funcion in filas:
    print(f"{nombre:28}{funcion(med['tavg']):>10.1f}{funcion(mad['tavg']):>10.1f}")
print(f"{'TMIN promedio (°C)':28}{np.nanmean(med['tmin']):>10.1f}{np.nanmean(mad['tmin']):>10.1f}")
print(f"{'PRCP total (mm)':28}{np.nansum(med['prcp']):>10.1f}{np.nansum(mad['prcp']):>10.1f}")
print(f"{'PRCP máxima en un día (mm)':28}{np.nanmax(med['prcp']):>10.1f}{np.nanmax(mad['prcp']):>10.1f}")
print(f"{'Días lluviosos':28}{resumen['Medellín']['dias_lluviosos']:>10}{resumen['Madrid']['dias_lluviosos']:>10}")
print(f"{'Días con PRCP reportada':28}{(~np.isnan(med['prcp'])).sum():>10}{(~np.isnan(mad['prcp'])).sum():>10}")

Como Medellín tiene 71 días sin dato de lluvia, el total anual no es justo. Comparo también solo en los 293 días con dato en las dos ciudades:

In [ ]:
print(f"Lluvia en días comunes: Medellín {med['prcp'][prcp_validas].sum():.1f} mm | Madrid {mad['prcp'][prcp_validas].sum():.1f} mm")
print(f"Días lluviosos comunes: Medellín {(med['prcp'][prcp_validas] >= UMBRAL_LLUVIA_MM).sum()} | "
      f"Madrid {(mad['prcp'][prcp_validas] >= UMBRAL_LLUVIA_MM).sum()}")
mas_calido = med["tavg"][tavg_validas] > mad["tavg"][tavg_validas]
print(f"Días en que Medellín fue más cálida que Madrid: {mas_calido.sum()} de {tavg_validas.sum()}")
meses_madrid_igual_o_mas = med["mes"][tavg_validas][~mas_calido]
print("Días en que Madrid igualó o superó a Medellín, por mes:", np.bincount(meses_madrid_igual_o_mas, minlength=13)[1:])

### Broadcasting

1. **Matriz de meses.** Comparo el vector de meses `(365,)` convertido a columna `(365, 1)` contra `np.arange(1, 13)` `(12,)`. NumPy estira las dos formas y sale una matriz booleana `(365, 12)`: cada columna es la máscara de un mes. Con eso saco los resúmenes mensuales sin escribir un `if` por mes.

In [ ]:
def resumen_mensual(d):
    meses_matriz = d["mes"][:, np.newaxis] == np.arange(1, 13)       # (365, 12)
    temp = np.where(meses_matriz, d["tavg"][:, np.newaxis], np.nan)  # (365, 1) se estira a (365, 12)
    lluvia = np.where(meses_matriz, d["prcp"][:, np.newaxis], np.nan)
    return {
        "forma_mascara": meses_matriz.shape,
        "tavg": np.nanmean(temp, axis=0),
        "prcp": np.nansum(lluvia, axis=0),
        "dias_lluviosos": (lluvia >= UMBRAL_LLUVIA_MM).sum(axis=0),
    }


mensual = {ciudad: resumen_mensual(d) for ciudad, d in datos.items()}
print("Forma de la máscara de meses:", mensual["Medellín"]["forma_mascara"])
print("         ", " ".join(f"{m:>6}" for m in MESES))
for ciudad, r in mensual.items():
    print(f"{ciudad[:8]:8} T", " ".join(f"{v:6.1f}" for v in r["tavg"]))
    print(f"{ciudad[:8]:8} P", " ".join(f"{v:6.1f}" for v in r["prcp"]))
    print(f"{ciudad[:8]:8} D", " ".join(f"{v:6d}" for v in r["dias_lluviosos"]))

2. **Anomalías.** Apilo las temperaturas mensuales en una matriz `(2, 12)` y le resto la media anual de cada ciudad como columna `(2, 1)`. La resta se aplica a los 12 meses de cada fila a la vez.

In [ ]:
temp_mensual = np.vstack([mensual["Medellín"]["tavg"], mensual["Madrid"]["tavg"]])   # (2, 12)
media_anual = np.array([[np.nanmean(med["tavg"])], [np.nanmean(mad["tavg"])]])        # (2, 1)
anomalias = temp_mensual - media_anual                                                 # (2, 12)

print("Formas:", temp_mensual.shape, "-", media_anual.shape, "=", anomalias.shape)
for ciudad, fila in zip(ESTACIONES, anomalias):
    print(f"{ciudad:9}", " ".join(f"{v:+5.1f}" for v in fila),
          f"| rango {fila.min():+.1f} a {fila.max():+.1f}")

En Medellín ningún mes se aleja más de 1,6 °C de su promedio anual; en Madrid enero queda 9,5 °C por debajo y julio 11,9 °C por encima.

3. **Cortes sobre los resúmenes mensuales.** Junio–agosto son las posiciones `5:8` y octubre–noviembre `9:11`.

In [ ]:
for ciudad, r in mensual.items():
    mitad = r["dias_lluviosos"][5:8].sum()
    final = r["dias_lluviosos"][9:11].sum()
    print(f"{ciudad}: días lluviosos jun–ago = {mitad} | oct–nov = {final} | mes con más días lluviosos: "
          f"{MESES[int(np.argmax(r['dias_lluviosos']))]} ({r['dias_lluviosos'].max()})")

## D. Gráficas

In [ ]:
COLORES = {"Medellín": "#1b9e77", "Madrid": "#d95f02"}
x = np.arange(12)

fig, ax = plt.subplots(figsize=(10, 4.5))
for ciudad, r in mensual.items():
    ax.plot(x, r["tavg"], marker="o", color=COLORES[ciudad], label=ciudad)
ax.set_title(f"Temperatura media mensual – {ANIO}")
ax.set_xlabel("Mes")
ax.set_ylabel("Temperatura media (°C)")
ax.set_xticks(x, MESES)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Qué veo:** Medellín es casi una línea plana entre 23,1 °C (octubre) y 25,7 °C (agosto). Madrid sube de 6,0 °C en enero a 27,4 °C en julio y vuelve a bajar. Las líneas se cruzan: en julio Madrid fue más cálida (27,4 frente a 24,8 °C) y en agosto quedaron casi iguales (25,6 frente a 25,7 °C), así que mi predicción de "Medellín más cálida todos los meses" falló.

**Qué no permite concluir:** es un solo año y una estación por ciudad; no dice si 2019 fue normal ni representa toda la ciudad. Además `TAVG` es la media diaria que reporta cada estación y puede calcularse con horas distintas en cada país, así que diferencias de 1 °C no las tomo como reales.

In [ ]:
ancho = 0.4
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - ancho / 2, mensual["Medellín"]["prcp"], ancho, color=COLORES["Medellín"], label="Medellín")
ax.bar(x + ancho / 2, mensual["Madrid"]["prcp"], ancho, color=COLORES["Madrid"], label="Madrid")
ax.set_title(f"Precipitación acumulada por mes – {ANIO} (solo días con dato)")
ax.set_xlabel("Mes")
ax.set_ylabel("Precipitación (mm)")
ax.set_xticks(x, MESES)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Qué veo:** Medellín acumula más lluvia en 10 de los 12 meses; sus picos son abril (169,2 mm), noviembre (154,9 mm) y marzo (139,7 mm). Madrid casi no llueve de mayo a julio (0,1 mm en mayo) y su mes más lluvioso es diciembre (90,1 mm). Agosto es el mes más seco de Medellín (6,1 mm) pero no de Madrid (58,4 mm, casi todo en tres días: 21, 26 y 30). En ambas abril es de los meses más lluviosos.

**Qué no permite concluir:** las barras de Medellín en mayo y julio están subestimadas porque faltan 12 y 13 días de datos; no puedo decir que mayo fuera seco. Tampoco puedo explicar por qué llueve más en un sitio que en otro: solo estoy observando acumulados.

In [ ]:
bins = np.arange(-2, 27, 1)
fig, ax = plt.subplots(figsize=(10, 4.5))
for ciudad, d in datos.items():
    tmin = d["tmin"][~np.isnan(d["tmin"])]
    ax.hist(tmin, bins=bins, alpha=0.6, color=COLORES[ciudad], label=f"{ciudad} ({len(tmin)} días)")
ax.set_title(f"Distribución de la temperatura mínima diaria – {ANIO}")
ax.set_xlabel("Temperatura mínima (°C)")
ax.set_ylabel("Número de días")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Qué veo:** casi todas las mínimas de Medellín quedan entre 16 y 20 °C (extremos 14,2 y 20,4); las de Madrid van de -1,8 a 25 °C. Las mañanas de Medellín son parecidas todo el año; en Madrid dependen mucho de la estación.

**Qué no permite concluir:** Medellín tiene 89 días sin `TMIN` y Madrid ninguno, por eso las alturas de las barras no se comparan directamente (cada ciudad tiene distinto total de días). Sirve para ver la forma y el rango, no para contar días exactos.

## E. Interpretación

**Respuesta a la pregunta guía.** En 2019 la mayor diferencia entre Medellín y Madrid fue la estabilidad de la temperatura, y la similitud fue el calendario de los meses secos y lluviosos.

**Hallazgo 1 – Diferencia: temperatura estable vs. estacional.**
- Media anual: Medellín 24,1 °C, Madrid 15,5 °C.
- Desviación estándar diaria: 1,6 °C en Medellín frente a 7,6 °C en Madrid.
- Rango de las medias mensuales: 2,6 °C en Medellín (23,1 a 25,7) frente a 21,4 °C en Madrid (6,0 a 27,4).
- Medellín fue más cálida en 302 de los 356 días comparables; los otros 54 caen entre mayo y septiembre, 25 de ellos en julio (ese mes Madrid promedió 27,4 °C frente a 24,8 °C).
- Con el mismo umbral de 25 °C: Medellín tuvo 105 días cálidos en 11 de los 12 meses (todos menos octubre); Madrid 55, todos entre junio y septiembre.

**Hallazgo 2 – Similitud: mitad de año con pocos días de lluvia y final de año más lluvioso.**
- En las dos ciudades llovió en menos días en junio–agosto que en octubre–noviembre: Medellín 14 frente a 24 días lluviosos; Madrid 7 frente a 18.
- Noviembre tuvo 12 días lluviosos en cada ciudad (en Madrid es su máximo; en Medellín empata con septiembre y octubre).
- Abril es de los meses más lluviosos en ambas (Medellín 169,2 mm, su máximo; Madrid 83,2 mm, su segundo mes).
- La similitud es de calendario, no de cantidad: en los mismos meses Medellín sigue recibiendo mucha más agua.

**Hallazgo 3 – Diferencia: cantidad de lluvia.** En los 293 días con dato en ambas, Medellín acumuló 1095,1 mm y Madrid 339,3 mm (3,2 veces más), con 91 días lluviosos frente a 46.

**Cómo afectan el periodo y los faltantes.**
- Uso el mismo periodo (todo 2019) y la misma frecuencia (diaria) en las dos tablas, así que las comparaciones mes a mes son válidas en tiempo.
- La lluvia de Medellín tiene 71 días sin dato; por eso el total anual (1107,0 mm) es un mínimo y mayo/julio están subestimados. Para la comparación justa usé solo los 293 días comunes.
- `TMIN` de Medellín tiene 89 faltantes; la uso solo para ver la forma de la distribución.
- Es un año y una estación por ciudad: no sirve para hablar del clima "normal" de ninguna.

**Sin causalidad.** Que Madrid sea seco en verano y Medellín estable no lo explico con estos datos: no medí altitud, latitud ni corrientes de aire. Solo describo lo que se observa en 2019.

### Revisión final
- [x] Una tabla de Medellín (Colombia) y otra de Madrid (España).
- [x] Mismo periodo (2019-01-01 a 2019-12-31) y misma frecuencia (diaria).
- [x] Fuentes, variables y unidades documentadas.
- [x] NumPy: arrays, cortes, máscaras y broadcasting.
- [x] Tres gráficas con título, ejes, unidades y leyenda.
- [x] Notebook en `EA3_comparacion_climatica/` del repositorio de la EA1.